#### diag_validation_mun_N.ipynb

**Author:** James Sayre
**Email:** jsayre@ucdavis.edu
**Date Modified:** 2026-08-14

**Description:** Diagnose why $N$ varies across models in Table 1 (`validation_mun_level_2022.tex`).
For each prediction file behind the table, report the years covered, the municipality-years
available, and decompose the evaluation N (full-sample CV vs. 20% held-out municipalities,
multi-year vs. 2022-only).

**Inputs:** prediction parquets in `~/Dropbox/Projects/Maize_prediction/Data/predictions/`,
AEF ADC universe, SIAP Spring-Summer maize yields
**Outputs:** none (diagnostic printout only)

In [1]:
import os
import numpy as np
import pandas as pd

# ── Directories ──────────────────────────────────────
home_dir   =  os.path.expanduser("~")
proj_dir   =  os.path.join(home_dir, "Dropbox", "Projects", "Maize_prediction")
aef_dir    =  os.path.join(proj_dir, "Data", "alpha_earth")
pred_dir   =  os.path.join(proj_dir, "Data", "predictions")

# ── Inputs ─────────────────────────────────────────
aef_file   =  os.path.join(aef_dir, "alpha_earth_mex_adcs.parquet")   # AEF ADC-year universe
siap_path  =  os.path.join(home_dir, "Dropbox", "Projects",
                           "The Promise of Crop Substitution",
                           "data", "SIAP", "Cleaned",
                           "siap_ag_prod_estimation_by_season.dta")   # SIAP mun-season yields

# prediction files exactly as listed in validation_mun_level.py MODELS
MODELS  =  [
    ("NDVI Hist.",    "mun_harmonic_h3_fixed_gb_preds.parquet",    "yield_pred",       "mun"),
    ("NDVI Q-Hist.",  "mun_harmonic_h3_quantile_gb_preds.parquet", "yield_pred",       "mun"),
    ("AEF mean",      "adc_alpha_earth_preds_maize.parquet",       "yield_pred",       "adc"),
    ("Agg-NN",        "adc_agg_nn_preds_maize_phase2.parquet",     "yield_pred_agg_nn", "adc"),
    ("AEF Hist",      "adc_aef_hist_gb_preds.parquet",             "yield_pred",       "adc"),
    ("AEF Hist Ens.", "adc_aef_hist_ens_preds.parquet",            "pred",             "adc"),
]
SEED  =  42

In [2]:
#### Reproduce the SIAP labels and the 80/20 municipality split (mirrors validation_mun_level.py)
aef            =  pd.read_parquet(aef_file, columns=['adcid', 'year'])
aef['muncode'] =  aef['adcid'].str[:5]

siap            =  pd.read_stata(siap_path)
siap['muncode'] =  siap['muncode'].apply(lambda x: str(int(x)).zfill(5))
siap['yield']   =  siap['q'] / siap['ha_planted']
sm  =  siap[(siap['name'] == 'Maize') &
            (siap['growing_season'] == 'Spring-Summer') &
            (siap['year'] >= 2017)][['muncode', 'year', 'yield']]

valid  =  sorted(set(zip(aef['muncode'], aef['year'])) & set(zip(sm['muncode'], sm['year'])))
umuns  =  sorted(set(k[0] for k in valid))
np.random.seed(SEED)
np.random.shuffle(umuns)
val_muns  =  set(umuns[int(0.8 * len(umuns)):])
print(f"SIAP S-S maize mun-years 2017+: {len(sm):,};  municipalities in split: {len(umuns):,} "
      f"({len(val_muns):,} held out)")

SIAP S-S maize mun-years 2017+: 18,625;  municipalities in split: 2,361 (473 held out)


In [3]:
#### Decompose N per model row
rows  =  []
for label, f, col, level in MODELS:
    p  =  pd.read_parquet(os.path.join(pred_dir, f))
    has_year  =  'year' in p.columns
    if not has_year:
        p['year']  =  2022
    p  =  p.dropna(subset=[col])
    if level == 'mun':
        p['muncode']  =  p['muncode'].astype(str).str.zfill(5)
        my  =  p[['muncode', 'year']].drop_duplicates()
    else:
        p['muncode']  =  p['adcid'].astype(str).str[:5]
        my  =  p[['muncode', 'year']].drop_duplicates()
    yrs      =  f"{int(my['year'].min())}-{int(my['year'].max())}" + ("" if has_year else " (no year col)")
    joined   =  my.merge(sm, on=['muncode', 'year'], how='inner')
    n_full   =  len(joined)                                      # full-sample N (all muni-years)
    n_hold   =  len(joined[joined['muncode'].isin(val_muns)])    # 20% held-out muns, all years
    j22      =  joined[joined['year'] == 2022]
    rows.append([label, yrs, n_full, n_hold, len(j22[j22['muncode'].isin(val_muns)])])

diag  =  pd.DataFrame(rows, columns=['model', 'years_in_file', 'N_full_sample',
                                     'N_heldout_allyrs', 'N_heldout_2022'])
print(diag.to_string(index=False))

        model years_in_file  N_full_sample  N_heldout_allyrs  N_heldout_2022
   NDVI Hist.     2017-2024          18428              3723             464
 NDVI Q-Hist.     2017-2024          18428              3723             464
     AEF mean     2017-2024          18486              3734             465
       Agg-NN     2017-2024          18486              3734             465
     AEF Hist     2017-2024          18474              3734             465
AEF Hist Ens.     2022-2022           2228               450             450


In [4]:
#### Which sample does each Table-1 row actually use?
print("Table 1 uses: N_full_sample for NDVI rows (mun-level CV over ALL muni-years),")
print("N_heldout_allyrs for AEF mean / Agg-NN / AEF Hist (20% held-out muns, 2017-2024),")
print("N_heldout_2022 for AEF Hist Ens. (2022-only prediction file).")

Table 1 uses: N_full_sample for NDVI rows (mun-level CV over ALL muni-years),
N_heldout_allyrs for AEF mean / Agg-NN / AEF Hist (20% held-out muns, 2017-2024),
N_heldout_2022 for AEF Hist Ens. (2022-only prediction file).


In [5]:
#### Do the AEF Hist Ensemble's feature inputs already cover 2017-2024?
# gb_aef_hist_ensemble.py filters its ADC feature files to year == 2022 at load
# time; check whether the parquets on disk actually contain the other years.
ens_inputs  =  [
    "alpha_earth_mex_adcs_binned_hist.parquet",   # ADC binned histograms (bins model test set)
    "alpha_earth_mex_adcs_hist.parquet",          # ADC percentile features (pct model test set)
    "alpha_earth_mex_adcs.parquet",               # ADC mean embeddings
    "alpha_earth_mex_mun_binned_hist.parquet",    # mun binned histograms (training)
    "alpha_earth_mex_mun_hist.parquet",           # mun percentile features (training)
]
for f in ens_inputs:
    yrs  =  pd.read_parquet(os.path.join(aef_dir, f), columns=['year'])['year']
    per  =  yrs.value_counts().sort_index()
    print(f"{f:45s} {int(yrs.min())}-{int(yrs.max())}  "
          f"rows/yr min={per.min():,} max={per.max():,}  n_years={len(per)}")

alpha_earth_mex_adcs_binned_hist.parquet      2017-2024  rows/yr min=289,191 max=295,184  n_years=8


alpha_earth_mex_adcs_hist.parquet             2017-2024  rows/yr min=292,821 max=295,184  n_years=8
alpha_earth_mex_adcs.parquet                  2017-2024  rows/yr min=126,335 max=126,335  n_years=8
alpha_earth_mex_mun_binned_hist.parquet       2017-2024  rows/yr min=2,456 max=2,456  n_years=8


alpha_earth_mex_mun_hist.parquet              2017-2024  rows/yr min=2,456 max=2,456  n_years=8
